# Chapter 12: The Tool Gateway — Fault Injection and Interception

**Learning objectives:**
- Install a `ToolGateway` on a mock agent and inspect the `GatewayTranscript`
- Create `FaultProfile` objects for blocking, delaying and mocking tools
- Observe how different recovery modes (`retry`, `fallback`, `escalate`) handle blocked tools
- Understand the three-tier gateway decision pipeline: policy, fault injection, GMS plausibility

---
## Preamble

Chapter 11 introduced the adapter layer that wraps any SUT behind a uniform `evaluate()` interface. The adapter captures tool calls, latency and errors — but it treats the tools as a fixed environment. In practice, testing requires *control* over that environment: blocking tools to test recovery, injecting latency to test SLA resilience, substituting mock responses to isolate agent logic from tool behavior.

The **ToolGateway** provides this control. It intercepts every tool call through the harness hook pipeline and applies a configurable decision:

| Decision | Effect |
|---|---|
| `allow` | Tool executes normally |
| `block` / `inject_fault` | Tool is blocked; agent must recover |
| `mock` | Tool is intercepted; a mock response is injected |
| `stale_data` | Tool is intercepted; a cached previous result is substituted |

Every decision is recorded in a `GatewayTranscript` for audit.

In [ ]:
# === Preamble (all notebooks) ===
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
from collections import Counter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
from importlib.resources import files
# Load the Chapter 4 store (cap-on fixture from Step 2)
from knowlytix.harness.testing import DOEGMSBenchmark, DOEHarnessConfig
import math

config = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md")),
    stores_dir="../lab_stores",
    store_name="ch04_ground_truth",
    n_runs=4,
    ingest_epochs=500,
    enable_tracing=False,
    device=str(DEVICE),
)
assert config.enable_tracing is False

bench = DOEGMSBenchmark(config)
store = bench.ingest()

# Calibrate the GMS judge AND copy thresholds onto judge.config
# (Ch 6 canonical wiring; required for cap-on stores).
bench.judge.calibrate(seed=42)
thr = bench.judge._thresholds or {}
if thr.get("tau_contra") is not None:
    bench.judge.config.tau_contra = float(thr["tau_contra"])
if thr.get("tau_ent") is not None:
    bench.judge.config.tau_ent = float(thr["tau_ent"])
if thr.get("holonomy") is not None:
    bench.judge.config.tau_path = float(thr["holonomy"])
if bench.judge.config.tau_contra is None:
    bench.judge.config.tau_contra = math.sqrt(2.0) + 0.2
if bench.judge.config.tau_ent is None:
    bench.judge.config.tau_ent = math.sqrt(2.0) / 2.0
if bench.judge.config.tau_path is None:
    bench.judge.config.tau_path = 2.5

print(f"Store: {store.stats()['entities']} entities, "
      f"{store.stats()['triples']} triples, "
      f"{store.stats()['enm_entries']} ENM")

---

## FaultProfile: Defining Fault Injection Rules

A `FaultProfile` specifies what the gateway does when a tool call matches its pattern. Profiles are matched by `fnmatch` glob against the tool name — the first matching profile wins.

**FaultProfile fields:**

| Field | Type | Default | Purpose |
|---|---|---|---|
| `tool_pattern` | str | `"*"` | fnmatch glob (e.g. `"risk_check"`, `"modeva_*"`) |
| `latency_ms` | float | 0.0 | Artificial delay before allowing |
| `error_rate` | float | 0.0 | Probability [0,1] of blocking with error |
| `error_message` | str | `"Simulated tool failure"` | Message on injected error |
| `stale_data` | bool | False | Cache and replay previous results |
| `mock_response` | str or None | None | If set, intercept and return this response |
| `schema_strict` | bool | True | Reserved for schema validation |

In [ ]:
# Create three fault profiles demonstrating different injection types
from knowlytix.harness.testing.gateway import FaultProfile

# 1. Block escalation with 100% error rate
block_escalation = FaultProfile(
    tool_pattern="submit_escalation",
    error_rate=1.0,
    error_message="Escalation service unavailable",
)

# 2. Mock the risk_check tool with a canned response
mock_risk = FaultProfile(
    tool_pattern="risk_check",
    mock_response="MOCK: Risk check unavailable during maintenance",
)

# 3. Allow data_lookup but inject 50ms latency
slow_lookup = FaultProfile(
    tool_pattern="data_lookup",
    latency_ms=50,
)

profiles = [block_escalation, mock_risk, slow_lookup]
for p in profiles:
    print(f"Pattern: {p.tool_pattern:25s}  "
          f"error_rate={p.error_rate}  "
          f"mock={'yes' if p.mock_response else 'no':3s}  "
          f"latency={p.latency_ms}ms")

In [ ]:
# YAML serialization round-trip
import yaml

yaml_data = {"fault_profiles": [p.to_dict() for p in profiles]}
yaml_str = yaml.dump(yaml_data, default_flow_style=False, sort_keys=False)
print("=== Fault Profiles YAML ===")
print(yaml_str)

---

## GatewayDecision and GatewayTranscript

Every interception is recorded as a `GatewayDecision`. The full sequence for a run is collected in a `GatewayTranscript`.

**GatewayDecision fields:**

| Field | Type | Purpose |
|---|---|---|
| `tool_name` | str | Name of the intercepted tool |
| `action` | str | `allow`, `block`, `mock`, `stale_data`, or `inject_fault` |
| `reason` | str | Human-readable explanation |
| `gms_score` | float or None | GMS plausibility score for the tool call |
| `mock_plausibility_geodesic` | float or None | Geodesic distance for mock realism |
| `timestamp` | str | ISO timestamp (auto-filled) |

**GatewayTranscript fields:**

| Field | Type | Purpose |
|---|---|---|
| `run_id` | str | Identifier for the benchmark run |
| `decisions` | list[GatewayDecision] | Chronological list of all decisions |

---

## The ToolGateway Decision Pipeline

The gateway hooks into the harness at two interception points:

1. **`before_tool`** — called before execution. Can block the call entirely (`inject_fault`) or allow with latency.
2. **`intercept_tool`** — called after `before_tool` allows. Can substitute a mock response or stale cached data.

The decision pipeline runs in order:
1. Policy engine check (if configured) — can `block`
2. Error injection — probabilistic `inject_fault` from fault profiles
3. Latency injection — delay then allow
4. Mock/stale data interception — substitute response via `intercept_tool`
5. GMS plausibility scoring — informational annotation on `allow` decisions

> **Note:** `FullMockHarness` supports both `before_tool` and `intercept_tool` hooks. The basic `MockHarness` only supports `before_tool`, so mock response injection requires `FullMockHarness`.

---

## Lab 12.1 — Inject Faults and Observe Recovery

In this lab we install a gateway on a `FullMockHarness`, inject faults on two tools and inspect the gateway transcript to see what happened.

In [ ]:
from importlib.resources import files
# Lab 12.1a — Install gateway with error injection on risk_check
from knowlytix.harness.testing.gateway import ToolGateway, FaultProfile
from knowlytix.harness.fixtures import FullMockHarness

# Block risk_check with 100% error rate
profiles = [
    FaultProfile(
        tool_pattern="risk_check",
        error_rate=1.0,
        error_message="Risk service unavailable (simulated)",
    ),
]

gateway = ToolGateway(store=store, fault_profiles=profiles, run_id="lab12-1a")
harness = FullMockHarness(recovery_mode="fallback")
gateway.install(harness)

# Run a question that triggers data_lookup + risk_check
context = open(str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md"))).read()
prompt = (
    f"<report>\n{context}\n</report>\n\n"
    "Answer the following question using the report and any tools "
    "available to you.\n\n"
    "What is the risk rating of MDL-001?"
)
answer = harness.run(prompt)
print(f"Answer: {answer!r}")

# Inspect the transcript
transcript = gateway.get_transcript()
print(f"\n=== GatewayTranscript ({len(transcript.decisions)} decisions) ===")
for i, d in enumerate(transcript.decisions):
    print(f"\nDecision {i+1}:")
    print(f"  tool_name : {d.tool_name}")
    print(f"  action    : {d.action}")
    print(f"  reason    : {d.reason}")
    if d.gms_score is not None:
        print(f"  gms_score : {d.gms_score:.4f}")

gateway.uninstall(harness)

In [ ]:
# Lab 12.1b — Mock response injection via intercept_tool
profiles_mock = [
    FaultProfile(
        tool_pattern="risk_check",
        mock_response="MOCK: Risk=Critical, Status=Under Review",
    ),
]

gateway_mock = ToolGateway(store=store, fault_profiles=profiles_mock, run_id="lab12-1b")
harness_mock = FullMockHarness(recovery_mode="fallback")
gateway_mock.install(harness_mock)

answer_mock = harness_mock.run(prompt)
print(f"Answer with mock injection: {answer_mock!r}")

transcript_mock = gateway_mock.get_transcript()
print(f"\n=== GatewayTranscript ({len(transcript_mock.decisions)} decisions) ===")
for i, d in enumerate(transcript_mock.decisions):
    print(f"\nDecision {i+1}:")
    print(f"  tool_name : {d.tool_name}")
    print(f"  action    : {d.action}")
    print(f"  reason    : {d.reason}")
    if d.mock_plausibility_geodesic is not None:
        print(f"  mock_plausibility_geodesic: {d.mock_plausibility_geodesic:.4f}")
    if d.gms_score is not None:
        print(f"  gms_score : {d.gms_score:.4f}")

gateway_mock.uninstall(harness_mock)

In [ ]:
# Lab 12.1c — No faults: baseline with gateway observing only
gateway_clean = ToolGateway(store=store, fault_profiles=[], run_id="lab12-1c")
harness_clean = FullMockHarness(recovery_mode="fallback")
gateway_clean.install(harness_clean)

answer_clean = harness_clean.run(prompt)
print(f"Answer (no faults): {answer_clean!r}")

transcript_clean = gateway_clean.get_transcript()
print(f"\n=== Clean Transcript ({len(transcript_clean.decisions)} decisions) ===")
for i, d in enumerate(transcript_clean.decisions):
    print(f"  {d.tool_name:20s} {d.action:12s} gms={d.gms_score}")

gateway_clean.uninstall(harness_clean)

---

## Lab 12.2 — Compare Recovery Modes

The `FullMockHarness` supports three recovery modes when a tool is blocked:

| Mode | Behavior |
|---|---|
| `retry` | Notes the blockage (no recovery action) |
| `fallback` | Falls back to `recall_memory` |
| `escalate` | Calls `submit_escalation` to flag the issue |

We run the same fault injection scenario under all three modes and compare the answers and tool call patterns.

In [ ]:
# Lab 12.2 — Compare recovery modes under fault injection
# Block both risk_check and recall_memory to stress escalation path
fault_profiles = [
    FaultProfile(
        tool_pattern="risk_check",
        error_rate=1.0,
        error_message="Risk service unavailable",
    ),
]

print("=== Recovery Mode Comparison ===")
print(f"{'Mode':<12s} {'Answer':<50s} {'Decisions':>9s}")
print("-" * 75)

for mode in ["retry", "fallback", "escalate"]:
    gw = ToolGateway(store=store, fault_profiles=fault_profiles,
                     run_id=f"lab12-2-{mode}")
    h = FullMockHarness(recovery_mode=mode)
    gw.install(h)

    ans = h.run(prompt)
    tr = gw.get_transcript()

    # Count decisions by action
    actions = Counter(d.action for d in tr.decisions)
    action_str = ", ".join(f"{k}={v}" for k, v in sorted(actions.items()))

    print(f"{mode:<12s} {ans[:48]!r:<50s} {action_str}")
    gw.uninstall(h)

In [ ]:
# Lab 12.2b — Detailed transcript for each mode
for mode in ["retry", "fallback", "escalate"]:
    gw = ToolGateway(store=store, fault_profiles=fault_profiles,
                     run_id=f"lab12-2b-{mode}")
    h = FullMockHarness(recovery_mode=mode)
    gw.install(h)

    ans = h.run(prompt)
    tr = gw.get_transcript()

    print(f"\n=== mode={mode!r} — {len(tr.decisions)} decisions ===")
    for d in tr.decisions:
        score_str = f"gms={d.gms_score:.4f}" if d.gms_score is not None else ""
        print(f"  [{d.action:12s}] {d.tool_name:<22s} {d.reason[:50]}  {score_str}")
    print(f"  Answer: {ans[:80]!r}")

    gw.uninstall(h)

---

## Gateway with AgentAdapter

The gateway integrates cleanly with the `AgentAdapter` from Chapter 11. The adapter's constructor accepts an optional `gateway` parameter — when provided, the gateway is automatically installed before each evaluation and uninstalled afterward.

In [ ]:
# Gateway + AgentAdapter integration
from knowlytix.harness.testing.adapters import AgentAdapter

fault_profiles_adapter = [
    FaultProfile(tool_pattern="risk_check", error_rate=1.0,
                 error_message="Risk service timeout"),
]
gateway_adapter = ToolGateway(
    store=store, fault_profiles=fault_profiles_adapter,
    run_id="lab12-adapter",
)

# Pass gateway to AgentAdapter
adapter = AgentAdapter(
    harness=FullMockHarness(recovery_mode="fallback"),
    gateway=gateway_adapter,
)

trace = adapter.evaluate(
    question="What is the risk rating of MDL-001?",
    context=context,
)

print("=== AgentAdapter + Gateway ===")
print(f"Answer     : {trace.raw_output!r}")
print(f"Tool calls : {len(trace.tool_calls)}")
print(f"Latency    : {trace.latency_ms:.1f} ms")
for call in trace.tool_calls:
    print(f"  -> {call.tool_name}({call.arguments}) = {call.result[:60]!r}")

# Gateway transcript is populated even though uninstall happened
tr = gateway_adapter.get_transcript()
print(f"\nTranscript: {len(tr.decisions)} decisions")
for d in tr.decisions:
    print(f"  [{d.action:12s}] {d.tool_name}")

---

## End-to-End: 520 Questions with Fault Injection

We run the full DOE benchmark through the adapter with fault injection enabled, then analyze how faults affect the gateway transcript and answer distribution.

In [ ]:
# Run 640 DOE questions with fault injection
import random
random.seed(42)  # for reproducible error_rate sampling

e2e_profiles = [
    FaultProfile(tool_pattern="risk_check", error_rate=0.3,
                 error_message="Risk service degraded"),
]
e2e_gateway = ToolGateway(store=store, fault_profiles=e2e_profiles,
                          run_id="lab12-e2e")
e2e_adapter = AgentAdapter(
    harness=FullMockHarness(recovery_mode="fallback"),
    gateway=e2e_gateway,
)

doe_questions = bench.generate_questions()
traces = []
for dq in doe_questions:
    t = e2e_adapter.evaluate(
        question=dq.question.natural_language,
        context=context,
    )
    traces.append(t)

print(f"Collected {len(traces)} traces")
print(f"Errors   : {sum(1 for t in traces if t.error)}")
print(f"Non-empty: {sum(1 for t in traces if t.raw_output)}")

In [ ]:
# Analyze the gateway transcript
tr_e2e = e2e_gateway.get_transcript()
print(f"=== Gateway Transcript Summary ({len(tr_e2e.decisions)} decisions) ===")

action_counts = Counter(d.action for d in tr_e2e.decisions)
for action, count in action_counts.most_common():
    print(f"  {action:15s} {count:>5d}  ({100*count/len(tr_e2e.decisions):.1f}%)")

# By tool
print("\nBy tool:")
tool_action = {}
for d in tr_e2e.decisions:
    key = (d.tool_name, d.action)
    tool_action[key] = tool_action.get(key, 0) + 1

for (tool, action), count in sorted(tool_action.items()):
    print(f"  {tool:20s} {action:15s} {count:>5d}")

In [ ]:
# GMS plausibility scores from allowed decisions
gms_scores = [d.gms_score for d in tr_e2e.decisions
              if d.action == "allow" and d.gms_score is not None]

if gms_scores:
    print("=== GMS Plausibility Scores (allowed decisions) ===")
    print(f"Count  : {len(gms_scores)}")
    print(f"Mean   : {np.mean(gms_scores):.4f}")
    print(f"Median : {np.median(gms_scores):.4f}")
    print(f"Min    : {np.min(gms_scores):.4f}")
    print(f"Max    : {np.max(gms_scores):.4f}")
else:
    print("No GMS plausibility scores recorded (store may not have tool relations)")

# Tool call distribution in traces (captures recovery tool calls too)
all_tools = Counter(c.tool_name for t in traces for c in t.tool_calls)
print("\n=== Tool Call Distribution in Traces ===")
for tool, count in all_tools.most_common():
    print(f"  {tool:22s} {count:>5d}")

---

## Exercises

**Exercise 12.1** — Create a `FaultProfile` with `stale_data=True` for `data_lookup`. Run two questions through the gateway: the first populates the cache, the second receives the stale result. Verify the second decision has `action="stale_data"` in the transcript.

**Exercise 12.2** — Write a YAML file with three fault profiles (one blocking, one mocking, one with latency). Load it with `load_fault_profiles()` and verify the profiles match your YAML.

**Exercise 12.3** — Run the 520-question benchmark with `error_rate=0.5` on `data_lookup` (the primary tool) under all three recovery modes. Compare the pass rates. Which mode degrades most gracefully?

---

## Summary

This chapter introduced the **ToolGateway** — the control layer that sits between the agent and its tools:

1. **FaultProfile** defines injection rules: error rates, mock responses, latency and stale data caching. Profiles are matched by fnmatch glob against tool names.

2. **ToolGateway** hooks into the harness at `before_tool` (blocking/error injection) and `intercept_tool` (mock/stale data substitution). Every decision is annotated with GMS plausibility scores.

3. **GatewayTranscript** records every interception decision for audit — action, reason, GMS score and timestamp.

4. **Recovery modes** determine how the agent handles blocked tools: `retry` (passive), `fallback` (memory recall) and `escalate` (human escalation).

In the next chapter we add **PolicyEngine** — a GMS-backed rule enforcement layer that governs which tools the agent may call and under what conditions.